# EDA пользовательского поведения в мобильном банке

Исследуем event-based поведение и драйверы конверсии в premium.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

## Шаг 1: Загрузка и первичный осмотр

In [ ]:
df = pd.read_csv('mobile_events.csv')
print('Размер:', df.shape)
display(df.head())
print(df.dtypes)
print('Пропуски:
', df.isna().sum())
print('Дубликаты:', df.duplicated().sum())
df['timestamp'] = pd.to_datetime(df['timestamp'])

## Шаг 2: Метрики активности

In [ ]:
df['date'] = df['timestamp'].dt.date
df['week'] = df['timestamp'].dt.to_period('W').astype(str)
df['month'] = df['timestamp'].dt.to_period('M').astype(str)

dau = df.groupby('date')['user_id'].nunique()
wau = df.groupby('week')['user_id'].nunique()
mau = df.groupby('month')['user_id'].nunique()
print('Средний DAU:', round(dau.mean(), 2))
print('WAU:
', wau)
print('MAU:
', mau)

plt.figure(figsize=(12,5))
dau.plot(marker='o')
plt.title('DAU по дням')
plt.xlabel('Дата')
plt.ylabel('Уникальные пользователи')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

events_per_user = df.groupby('user_id').size()
fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].hist(events_per_user, bins=30)
axes[0].set_title('Распределение событий на пользователя')
sns.boxplot(x=events_per_user, ax=axes[1])
axes[1].set_title('Boxplot событий на пользователя')
plt.tight_layout()
plt.show()

## Шаг 3: Анализ событий

In [ ]:
event_counts = df['event_type'].value_counts()
event_share = (event_counts / event_counts.sum() * 100).round(2)
display(pd.DataFrame({'count': event_counts, 'share_%': event_share}).head(10))

screen_dist = df['screen'].value_counts().to_frame('count')
screen_dist['share_%'] = (screen_dist['count'] / screen_dist['count'].sum() * 100).round(2)
display(screen_dist)

plt.figure(figsize=(8,5))
sns.barplot(x=screen_dist.index, y=screen_dist['count'])
plt.title('Распределение экранов')
plt.xlabel('Экран')
plt.ylabel('Количество событий')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Шаг 4: Воронка баннер -> клик -> подписка

In [ ]:
step1 = set(df.loc[df['event_type'] == 'view_banner', 'user_id'])
step2 = set(df.loc[df['event_type'] == 'click_banner', 'user_id'])
step3 = set(df.loc[df['event_type'] == 'sub_premium', 'user_id'])
funnel = pd.DataFrame({
    'stage': ['view_banner', 'click_banner', 'sub_premium'],
    'users': [len(step1), len(step1 & step2), len(step1 & step2 & step3)]
})
funnel['conv_from_prev_%'] = [
    np.nan,
    round(funnel.loc[1, 'users'] / max(funnel.loc[0, 'users'], 1) * 100, 2),
    round(funnel.loc[2, 'users'] / max(funnel.loc[1, 'users'], 1) * 100, 2)
]
display(funnel)

plt.figure(figsize=(8,5))
sns.barplot(data=funnel, x='stage', y='users')
plt.title('Воронка ключевого сценария')
plt.show()

## Шаг 5: Когортный анализ retention

In [ ]:
cohort_df = df.copy()
cohort_df['event_week'] = cohort_df['timestamp'].dt.to_period('W').apply(lambda x: x.start_time)
first_week = cohort_df.groupby('user_id')['event_week'].min().rename('cohort_week')
cohort_df = cohort_df.merge(first_week, on='user_id', how='left')
cohort_df['cohort_index'] = ((cohort_df['event_week'] - cohort_df['cohort_week']).dt.days // 7).astype(int)

cohort_pivot = cohort_df.groupby(['cohort_week', 'cohort_index'])['user_id'].nunique().reset_index().pivot(index='cohort_week', columns='cohort_index', values='user_id')
retention = cohort_pivot.divide(cohort_pivot[0], axis=0).loc[:, 0:4].fillna(0)

plt.figure(figsize=(12,6))
sns.heatmap(retention, annot=True, fmt='.0%', cmap='Blues')
plt.title('Retention heatmap')
plt.show()

## Шаг 6: Сегментация пользователей

In [ ]:
user_stats = df.groupby('user_id').agg(
    events_cnt=('event_type', 'count'),
    premium_flag=('package', lambda s: int((s == 'premium').max())),
    transfers=('event_type', lambda s: (s == 'transfer').sum()),
    age=('age', 'mean')
).reset_index()

q1, q2 = user_stats['events_cnt'].quantile([0.33, 0.66])
user_stats['segment'] = np.select(
    [user_stats['events_cnt'] <= q1, user_stats['events_cnt'] <= q2],
    ['Низкая', 'Средняя'],
    default='Высокая'
)
segment_summary = user_stats.groupby('segment').agg(
    users=('user_id', 'nunique'),
    premium_share=('premium_flag', 'mean'),
    avg_transfers=('transfers', 'mean'),
    avg_age=('age', 'mean')
).reset_index()
segment_summary['premium_share'] = (segment_summary['premium_share'] * 100).round(2)
display(segment_summary)

plot_df = segment_summary.melt(id_vars='segment', value_vars=['premium_share', 'avg_transfers', 'avg_age'])
plt.figure(figsize=(10,5))
sns.barplot(data=plot_df, x='segment', y='value', hue='variable')
plt.title('Сравнение сегментов')
plt.show()

## Шаг 7: Выводы

- Самая вовлекающая функция: `app_open`; среди продуктовых действий выделяются `view_banner` и платежи.
- Наибольшая потеря воронки: между `click_banner` и `sub_premium`.
- Самые ценные сегменты: пользователи с высокой активностью и большей долей premium.
- Рекомендации: упростить финальный шаг подписки и усилить персонализацию промо.

### Резюме для команды
EDA показывает стабильную активность в будни и просадку в выходные. Основная точка роста подписки — оптимизация шага после клика по баннеру.